# HerBERT + LoRA: klasyfikacja tekstu i tokenów

Dane z Zadania 1 (`rest-annotations.json`, 600 wypowiedzi). Klasyfikacja tekstu: `ton` (zwykły / agresywny), klasyfikacja tokenów: spany `Agresja` / `Kontekst` w schemacie BIO.

In [1]:
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification, AutoTokenizer

from lm_common import *

MODEL = "allegro/herbert-base-cased"
tok = AutoTokenizer.from_pretrained(MODEL)
train_df, val_df, test_df = load_data()
print(len(train_df), len(val_df), len(test_df))
train_df.ton.value_counts()

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


480 60 60


ton
zwykły       344
agresywny    136
Name: count, dtype: int64

## 1a. Klasyfikacja tekstu (BertForSequenceClassification)

In [2]:
seq_model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, id2label=dict(enumerate(TON)), label2id={l: i for i, l in enumerate(TON)})
print(type(seq_model).__name__)
seq_model = get_peft_model(seq_model, LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "key", "value"]))
seq_model.print_trainable_parameters()

seq_trainer = train(seq_model, tok, seq_dataset(train_df, tok), seq_dataset(val_df, tok), epochs=10, name="herbert-seq")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34682.39it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those para

BertForSequenceClassification
trainable params: 886,274 || all params: 125,330,692 || trainable%: 0.7071


Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map: 100%|██████████| 480/480 [00:00<00:00, 19921.29 examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map: 100%|██████████| 60/60 [00:00<00:00, 12910.85 examples/s]

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:498.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.637485,0.571245,0.716667,0.417476
2,0.546610,0.478877,0.716667,0.417476
3,0.381385,0.273855,0.883333,0.853710
4,0.276403,0.348470,0.833333,0.739583
5,0.235895,0.266468,0.900000,0.872159
6,0.202036,0.307310,0.883333,0.840849
7,0.178807,0.373391,0.866667,0.803601
8,0.174332,0.234514,0.933333,0.911111
9,0.149202,0.258628,0.916667,0.886321
10,0.132525,0.234970,0.916667,0.891265


## 1b. Klasyfikacja tokenów (BertForTokenClassification)

In [3]:
tok_model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS), id2label=dict(enumerate(TAGS)), label2id={l: i for i, l in enumerate(TAGS)})
print(type(tok_model).__name__)
tok_model = get_peft_model(tok_model, LoraConfig(task_type=TaskType.TOKEN_CLS, r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "key", "value"]))
tok_model.print_trainable_parameters()

tok_trainer = train(tok_model, tok, tok_dataset(train_df, tok), tok_dataset(val_df, tok), token_level=True, epochs=15, lr=5e-4, name="herbert-tok")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24741.82it/s]


[transformers] BertForTokenClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can b

BertForTokenClassification
trainable params: 888,581 || all params: 124,744,714 || trainable%: 0.7123


Epoch,Training Loss,Validation Loss,F1
1,0.634947,0.445494,0.000000
2,0.414920,0.399014,0.000000
3,0.362620,0.380727,0.000000
4,0.326433,0.376929,0.016129
5,0.298149,0.380705,0.051502
6,0.281225,0.380325,0.099644
7,0.262184,0.391955,0.115016
8,0.248745,0.401079,0.126183
9,0.232646,0.414541,0.153392
10,0.221997,0.415293,0.140575


## 1c. Ocena na zbiorze testowym

In [4]:
seq_preds = predict(seq_model, tok, test_df.text.tolist())
print(classification_report(test_df.label, seq_preds, target_names=TON, digits=3))

              precision    recall  f1-score   support

      zwykły      0.884     0.884     0.884        43
   agresywny      0.706     0.706     0.706        17

    accuracy                          0.833        60
   macro avg      0.795     0.795     0.795        60
weighted avg      0.833     0.833     0.833        60



In [5]:
spans_rep, tokens_rep = report_tokens(tok_trainer, tok_dataset(test_df, tok))
print("Ocena na poziomie spanów (dokładne dopasowanie, seqeval):")
print(spans_rep)
print("Ocena na poziomie tokenów:")
print(tokens_rep)

Ocena na poziomie spanów (dokładne dopasowanie, seqeval):
              precision    recall  f1-score   support

     Agresja      0.100     0.094     0.097        32
    Kontekst      0.104     0.143     0.120       119

   micro avg      0.103     0.132     0.116       151
   macro avg      0.102     0.118     0.108       151
weighted avg      0.103     0.132     0.115       151

Ocena na poziomie tokenów:
              precision    recall  f1-score   support

     Agresja      0.250     0.154     0.190        91
    Kontekst      0.357     0.335     0.345       412
           O      0.929     0.942     0.935      4224

    accuracy                          0.874      4727
   macro avg      0.512     0.477     0.490      4727
weighted avg      0.866     0.874     0.870      4727



In [6]:
for row in test_df.head(3).itertuples():
    print(row.text)
    print("złote:", [(s["labels"][0], s["text"]) for s in row.spans])
    print("pred: ", show_spans(row.text, tok, tok_model))
    print()

To też jest dobra wiadomość. Racjonalne państwo powinno po prostu korzystać z tych okazji, zwłaszcza że Polska potrzebuje dużych inwestycji nie tylko w zbrojenia, ale także w system energetyczny. Mamy system energetyczny na trytytki. Nie zbudowaliśmy ani jednej elektrowni jądrowej i za chwilę bardzo dużo może nas to po prostu kosztować. Proszę państwa, kiedy jest szansa, żeby nadrobić te zapóźnienia, wykorzystajmy te środki. Logiczne rozwiązanie jest takie: SAFE plus atom. Bierzemy środki z SAFE na obronność, a środki z NBP przeznaczamy na budowę elektrowni jądrowych i nadrabiamy zapóźnienia.
złote: [('Kontekst', 'Mamy system energetyczny'), ('Kontekst', 'wykorzystajmy te środki'), ('Kontekst', 'SAFE plus atom')]
pred:  [('Kontekst', 'dobra wiadomość'), ('Kontekst', 'nadrobić te zapóźnienia'), ('Kontekst', 'wykorzysta'), ('Kontekst', 'te środki'), ('Kontekst', 'nadrabiamy zapóźnienia')]

Drugi obszar projektu to ceny transferowe i informacja TPR. Zmienia się sposób raportowania. Inform

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:438.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


## 2. Przestrzeń wektorowa przykładów testowych (reprezentacja tokenu [CLS])

In [7]:
emb = embed(seq_model, tok, test_df.text.tolist(), pool="cls")
plot_space(emb, test_df.label, seq_preds, test_df.text, "HerBERT LoRA - CLS")

Dla porównania: model bazowy przed strojeniem (adapter wyłączony).

In [8]:
with seq_model.disable_adapter():
    base_emb = embed(seq_model, tok, test_df.text.tolist(), pool="cls")
plot_space(base_emb, test_df.label, seq_preds, test_df.text, "HerBERT bazowy - CLS")